# NB5 – Đuôi bài có chứa đoạn mở đầu của bài khác không? (kiểm tra trước khi dùng kết quả G2, R2-C3)
**Runtime:** CPU · High-RAM  ·  **Thời gian ước tính:** 10–15 phút

Notebook tự chứa: code được nhúng sẵn (các ô `%%writefile`), không cần upload file.
Có thể chạy **song song** với các notebook khác. Chọn *Runtime → Run all*.
Kết quả được chép vào `MyDrive/PLOS_revision/NB5_CPU_tail_teaser/`.

Lấy ngẫu nhiên 200 bài (seed 2026). So 60 từ cuối của mỗi bài với 60 từ đầu (tiêu đề + sapo) của toàn bộ 946.696 bài; bỏ qua các cặp là bài trùng nhau. Muốn ước lượng chặt hơn: đổi `N_SAMPLE` thành 5000.

In [ ]:
# 1) Drive (kết quả được chép vào đây) + 2) đăng nhập Hugging Face (tài khoản BaoNhan)
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
RESULT_DIR = "/content/drive/MyDrive/PLOS_revision/NB5_CPU_tail_teaser"
os.makedirs(RESULT_DIR, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # Colab secret tên HF_TOKEN (nếu có)
except Exception:
    from huggingface_hub import login
    login()                                                 # nếu chưa có secret: dán token vào ô hiện ra
from huggingface_hub import whoami
try:
    print("Đã đăng nhập Hugging Face:", whoami()["name"])
except Exception:
    raise SystemExit("CHƯA ĐĂNG NHẬP Hugging Face. Bấm biểu tượng chìa khóa (Secrets) bên trái, "
                     "bật 'Notebook access' cho HF_TOKEN, rồi chạy lại ô này.")
!pip -q install datasets pandas

## Code (nhúng sẵn – không cần sửa)

In [ ]:
%%writefile step5_tail_teaser.py
"""STEP 5 - Does the end of an article contain the opening of ANOTHER article?
(check before interpreting the head+tail gain of Reviewer #2, Concern 3)

For a random sample of articles (default 200, seed 2026) the last TAIL_WORDS words
are compared with the first LEAD_WORDS words (title + lead) of every other article
in BaoNhan/Vietnam_News_3_Sources_Plus, using 8-word shingles. A sampled article
"contains a teaser" when its tail shares >= MIN_HITS shingles with the opening of
another article that is NOT a duplicate of the sampled article itself.

Outputs (revision_results/): tail_teaser_sample.csv (one row per sampled article,
with the tail text for manual reading) and tail_teaser_report.txt.
CPU (High-RAM), about 10-15 minutes. Set N_SAMPLE=5000 for a tighter estimate.
"""
import os
import random
import re
import zlib
from pathlib import Path

import pandas as pd

K, TAIL_WORDS, LEAD_WORDS, MIN_HITS = 8, 60, 60, 3
SEED = 2026
WS = re.compile(r"\s+")
OUT = Path("revision_results")


def words(text, limit_chars=None):
    t = text or ""
    if limit_chars:
        t = t[:limit_chars]
    return WS.sub(" ", t).strip().lower().split(" ")


def shingles(ws):
    return {zlib.crc32(" ".join(ws[i:i + K]).encode("utf-8"))
            for i in range(max(0, len(ws) - K + 1))}


def tail_words(text):
    w = words(text)
    if len(w) < TAIL_WORDS + LEAD_WORDS:      # short article: tail = second half only
        return w[len(w) // 2:]
    return w[-TAIL_WORDS:]


def find_teasers(contents, sample_rows, progress=None):
    """contents: indexable sequence of article texts (whole corpus).
    Returns {row: (best_other_row or None, hits, lead_overlap_with_other)}."""
    tails = {r: shingles(tail_words(contents[r])) for r in sample_rows}
    leads = {r: shingles(words(contents[r], 3000)[:LEAD_WORDS]) for r in sample_rows}
    want = {}
    for r, sh in tails.items():
        for h in sh:
            want.setdefault(h, []).append(r)
    hits = {r: {} for r in sample_rows}
    n = len(contents)
    for other in range(n):
        for h in shingles(words(contents[other], 3000)[:LEAD_WORDS]):
            for r in want.get(h, ()):
                if r != other:
                    hits[r][other] = hits[r].get(other, 0) + 1
        if progress and other % 100000 == 0:
            progress(other, n)
    out = {}
    for r in sample_rows:
        cands = sorted(hits[r].items(), key=lambda kv: -kv[1])
        best = None
        for other, c in cands:
            if c < MIN_HITS:
                break
            overlap = len(leads[r] & shingles(words(contents[other], 3000)[:LEAD_WORDS]))
            if overlap >= MIN_HITS:          # same opening -> duplicate article, not a teaser
                continue
            best = (other, c, overlap)
            break
        out[r] = best if best else (None, cands[0][1] if cands else 0, 0)
    return out


if __name__ == "__main__":
    from datasets import load_dataset
    OUT.mkdir(exist_ok=True)
    ds = load_dataset("BaoNhan/Vietnam_News_3_Sources_Plus", split="train")
    contents = ds["content"]
    meta = ds.remove_columns(["content"]).to_pandas()
    n_sample = os.environ.get("N_SAMPLE", "200")
    rows = sorted(random.Random(SEED).sample(range(len(ds)), int(n_sample)))
    res = find_teasers(contents, rows, progress=lambda i, n: print(f"  scanned {i}/{n}", flush=True))
    recs = []
    for r in rows:
        other, h, ov = res[r]
        rec = dict(row=r, source=meta.source[r], label=meta.label[r], url=meta.url[r],
                   has_teaser=other is not None, shared_shingles=h,
                   other_label=meta.label[other] if other is not None else None,
                   other_url=meta.url[other] if other is not None else None,
                   tail_text=" ".join(tail_words(contents[r])))
        rec["same_label"] = rec["has_teaser"] and rec["other_label"] == rec["label"]
        recs.append(rec)
    df = pd.DataFrame(recs)
    df.to_csv(OUT / "tail_teaser_sample.csv", index=False)
    lines = [f"sampled articles: {len(df)} (seed {SEED}); tail = last {TAIL_WORDS} words, "
             f"lead = first {LEAD_WORDS} words, {K}-word shingles, >= {MIN_HITS} shared",
             f"tail contains the opening of another article: {df.has_teaser.sum()} "
             f"({100 * df.has_teaser.mean():.1f}%)"]
    t = df[df.has_teaser]
    if len(t):
        lines.append(f"  ... and that article has the SAME category: {t.same_label.sum()} "
                     f"({100 * t.same_label.mean():.1f}% of those)")
    lines.append("by source (share with teaser):")
    lines.append(df.groupby("source").has_teaser.agg(["size", "mean"]).round(3).to_string())
    lines.append("by category (share with teaser):")
    lines.append(df.groupby("label").has_teaser.agg(["size", "mean"]).round(3).to_string())
    (OUT / "tail_teaser_report.txt").write_text("\n".join(lines), encoding="utf-8")
    print("\n".join(lines))
    print("\nTen examples with a teaser (end of article):")
    for _, x in t.head(10).iterrows():
        print(f"- [{x.label} -> {x.other_label}] ...{x.tail_text[-220:]}")

## Dùng lại kết quả cũ trên Drive

In [ ]:
# Dùng lại kết quả của lần chạy trước trên Drive (nếu có) để bỏ qua các bước đã xong
import shutil, glob, os
for d in ["revision_data", "revision_data/preds", "revision_results"]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{RESULT_DIR}/{d}/*"):
        if os.path.isfile(f) and not os.path.exists(f"{d}/{os.path.basename(f)}"):
            shutil.copy(f, f"{d}/")
have = lambda p: os.path.exists(p)
print("preds:", len(glob.glob("revision_data/preds/*.npz")), "| corpus_index:", have("revision_data/corpus_index.pkl"),
      "| step2:", have("revision_results/step2_report.txt"))

## Chạy

In [ ]:
os.environ["N_SAMPLE"] = "200"
!python step5_tail_teaser.py

## Lưu kết quả

In [ ]:
# Chép kết quả sang Drive và tạo file zip để tải về
import shutil, glob
for d in ["revision_results", "revision_data"]:
    for f in glob.glob(f"{d}/*.txt") + glob.glob(f"{d}/*.csv") + glob.glob(f"{d}/*.tex") + \
             glob.glob(f"{d}/*.pdf") + glob.glob(f"{d}/*.tif") + glob.glob(f"{d}/*.pkl"):
        if os.path.getsize(f) < 500e6:
            os.makedirs(f"{RESULT_DIR}/{d}", exist_ok=True)
            shutil.copy(f, f"{RESULT_DIR}/{d}/")
os.makedirs(f"{RESULT_DIR}/revision_data/preds", exist_ok=True)
for f in glob.glob("revision_data/preds/*.npz"):
    shutil.copy(f, f"{RESULT_DIR}/revision_data/preds/")
print("Đã chép vào", RESULT_DIR)